# Gold-only model training

This notebook trains five Spark classifiers using only the gold training split. Logistic Regression and Gradient-Boosted Trees receive focused cross-validation. Model selection and calibration diagnostics use validation data only.


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy>=1.26.0",
    "pandas": "pandas>=2.1.0",
    "pyspark": "pyspark>=3.5.0,<4.0",
    "sklearn": "scikit-learn>=1.3.0",
}
missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages],
        check=True,
    )
    importlib.invalidate_caches()

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.ml.classification import (
    DecisionTreeClassifier, FMClassifier, GBTClassifier, LinearSVC,
    LogisticRegression, NaiveBayes, RandomForestClassifier
)
from pyspark.ml.evaluation import Evaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession, functions as F
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score
)

SEED = 42
BASE_DIR = Path("/kaggle/working/bangla_fake_review")
PROCESSED_DIR = BASE_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
AUDIT_DIR = BASE_DIR / "audit"
for directory in (MODELS_DIR, RESULTS_DIR, AUDIT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("BanglaFakeReview_GoldTraining")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def resolve_artifact_directory(
    preferred, required_names, source_label, input_root=Path("/kaggle/input")
):
    if all((preferred / name).exists() for name in required_names):
        return preferred
    candidates = []
    if input_root.exists():
        for first_artifact in input_root.rglob(required_names[0]):
            candidate = first_artifact.parent
            if all((candidate / name).exists() for name in required_names):
                candidates.append(candidate)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            f"Attach the saved output of {source_label} as a Kaggle input."
        )
    if len(candidates) > 1:
        raise RuntimeError(
            f"Multiple compatible outputs found for {source_label}: {candidates}"
        )
    return candidates[0]

stage_three_files = [
    "train_model_ready.parquet",
    "validation_model_ready.parquet",
]
INPUT_PROCESSED_DIR = resolve_artifact_directory(
    PROCESSED_DIR, stage_three_files, "notebook 03"
)
print(f"Reading notebook 03 artifacts from {INPUT_PROCESSED_DIR}")

train = spark.read.parquet(
    str(INPUT_PROCESSED_DIR / "train_model_ready.parquet")
)
validation = spark.read.parquet(
    str(INPUT_PROCESSED_DIR / "validation_model_ready.parquet")
)

label_counts = {
    int(row["label"]): int(row["count"])
    for row in train.groupBy("label").count().collect()
}
total_rows = sum(label_counts.values())
class_weights = {
    label: total_rows / (2.0 * count)
    for label, count in label_counts.items()
}
train = train.withColumn(
    "class_weight",
    F.when(F.col("label") == 0, F.lit(class_weights[0]))
    .otherwise(F.lit(class_weights[1])),
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 20:41:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading notebook 03 artifacts from /kaggle/input/notebooks/borshon47/featureengi/bangla_fake_review/processed


In [2]:
def macro_f1_from_frame(frame):
    counts = {
        (int(row["label"]), int(row["prediction"])): int(row["count"])
        for row in frame.groupBy("label", "prediction").count().collect()
    }
    class_f1 = []
    for label in (0, 1):
        true_positive = counts.get((label, label), 0)
        false_positive = sum(
            counts.get((other, label), 0) for other in (0, 1) if other != label
        )
        false_negative = sum(
            counts.get((label, other), 0) for other in (0, 1) if other != label
        )
        denominator = 2 * true_positive + false_positive + false_negative
        class_f1.append(2 * true_positive / denominator if denominator else 0.0)
    return float(sum(class_f1) / len(class_f1))


class MacroF1Evaluator(Evaluator):
    def _evaluate(self, dataset):
        return macro_f1_from_frame(dataset)

    def isLargerBetter(self):
        return True


def evaluate_predictions(predictions, model_name):
    columns = ["label", "prediction"]
    has_probability = "probability" in predictions.columns
    has_raw_prediction = "rawPrediction" in predictions.columns
    if has_probability:
        columns.append("probability")
    elif has_raw_prediction:
        columns.append("rawPrediction")

    rows = predictions.select(*columns).collect()
    labels = np.asarray([int(row["label"]) for row in rows])
    predicted = np.asarray([int(row["prediction"]) for row in rows])
    fake_target = (labels == 0).astype(int)

    if has_probability:
        fake_score = np.asarray([float(row["probability"][0]) for row in rows])
    else:
        fake_score = np.asarray([float(row["rawPrediction"][0]) for row in rows])

    result = {
        "model": model_name,
        "macro_f1": float(f1_score(labels, predicted, average="macro", zero_division=0)),
        "macro_precision": float(
            precision_score(labels, predicted, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(labels, predicted, average="macro", zero_division=0)
        ),
        "accuracy": float(accuracy_score(labels, predicted)),
        "fake_pr_auc": float(average_precision_score(fake_target, fake_score)),
        "fake_roc_auc": float(roc_auc_score(fake_target, fake_score)),
        "fake_precision": float(
            precision_score(labels, predicted, labels=[0], average="macro", zero_division=0)
        ),
        "fake_recall": float(
            recall_score(labels, predicted, labels=[0], average="macro", zero_division=0)
        ),
        "fake_f1": float(
            f1_score(labels, predicted, labels=[0], average="macro", zero_division=0)
        ),
        "genuine_precision": float(
            precision_score(labels, predicted, labels=[1], average="macro", zero_division=0)
        ),
        "genuine_recall": float(
            recall_score(labels, predicted, labels=[1], average="macro", zero_division=0)
        ),
        "genuine_f1": float(
            f1_score(labels, predicted, labels=[1], average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
    }
    return result


macro_f1_evaluator = MacroF1Evaluator()


In [3]:
logistic = LogisticRegression(
    featuresCol="features_linear",
    labelCol="label",
    weightCol="class_weight",
    maxIter=100,
    family="binomial",
)
logistic_grid = (
    ParamGridBuilder()
    .addGrid(logistic.regParam, [0.01, 0.1, 1.0])
    .addGrid(logistic.elasticNetParam, [0.0, 0.5])
    .build()
)
logistic_cv = CrossValidator(
    estimator=logistic,
    estimatorParamMaps=logistic_grid,
    evaluator=macro_f1_evaluator,
    numFolds=3,
    seed=SEED,
    parallelism=1,
)
logistic_model = logistic_cv.fit(train).bestModel

naive_bayes_model = NaiveBayes(
    featuresCol="features_nb",
    labelCol="label",
    weightCol="class_weight",
    smoothing=1.0,
    modelType="multinomial",
).fit(train)

linear_svc_model = LinearSVC(
    featuresCol="features_linear",
    labelCol="label",
    weightCol="class_weight",
    maxIter=100,
    regParam=0.1,
).fit(train)

random_forest_model = RandomForestClassifier(
    featuresCol="features_tree",
    labelCol="label",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=5,
    featureSubsetStrategy="sqrt",
    seed=SEED,
).fit(train)

gbt = GBTClassifier(
    featuresCol="features_tree",
    labelCol="label",
    weightCol="class_weight",
    maxIter=50,
    maxDepth=5,
    minInstancesPerNode=5,
    seed=SEED,
)
gbt_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, [40, 70])
    .addGrid(gbt.maxDepth, [4, 6])
    .addGrid(gbt.stepSize, [0.05, 0.1])
    .build()
)
gbt_cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=gbt_grid,
    evaluator=macro_f1_evaluator,
    numFolds=3,
    seed=SEED,
    parallelism=1,
)
gbt_model = gbt_cv.fit(train).bestModel

decision_tree_model = DecisionTreeClassifier(
    featuresCol="features_tree", labelCol="label",
    weightCol="class_weight", maxDepth=8, minInstancesPerNode=5, seed=SEED,
).fit(train)

fm_model = FMClassifier(
    # FMClassifier in Kaggle Spark does not expose weightCol.
    featuresCol="features_linear", labelCol="label",
    factorSize=8, stepSize=0.05, regParam=0.01, maxIter=80, seed=SEED,
).fit(train)

models = {
    "logistic_regression": logistic_model,
    "naive_bayes": naive_bayes_model,
    "linear_svc": linear_svc_model,
    "random_forest": random_forest_model,
    "gbt": gbt_model,
    "decision_tree": decision_tree_model,
    "factorization_machine": fm_model,
}
model_types = {
    "logistic_regression": "logistic_regression",
    "naive_bayes": "naive_bayes",
    "linear_svc": "linear_svc",
    "random_forest": "random_forest",
    "gbt": "gbt",
    "decision_tree": "decision_tree",
    "factorization_machine": "factorization_machine",
}

for name, model in models.items():
    model.write().overwrite().save(str(MODELS_DIR / f"{name}_gold"))


26/09/20 20:41:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/20 20:41:55 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:03 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:05 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:05 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:06 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:07 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:08 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:08 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:09 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/20 20:42:10 WARN DAGScheduler: 

In [4]:
validation_results = []
for name, model in models.items():
    predictions = model.transform(validation)
    validation_results.append(evaluate_predictions(predictions, name))

validation_results_frame = pd.DataFrame(validation_results).sort_values(
    ["macro_f1", "fake_pr_auc"],
    ascending=False,
)
validation_results_frame.to_csv(
    RESULTS_DIR / "gold_validation_metrics.csv",
    index=False,
)

selected_name = str(validation_results_frame.iloc[0]["model"])
selection = {
    "selected_model": selected_name,
    "selected_model_type": model_types[selected_name],
    "selection_source": "gold validation split",
    "primary_metric": "macro_f1",
    "tie_break_metric": "fake_pr_auc",
}
with open(MODELS_DIR / "gold_selection.json", "w", encoding="utf-8") as handle:
    json.dump(selection, handle, indent=2)

logistic_predictions = logistic_model.transform(validation)
calibration_rows = logistic_predictions.select("label", "probability").collect()
calibration = pd.DataFrame({
    "label": [int(row["label"]) for row in calibration_rows],
    "probability_fake": [float(row["probability"][0]) for row in calibration_rows],
    "confidence": [float(max(row["probability"])) for row in calibration_rows],
})
calibration["prediction"] = (calibration["probability_fake"] < 0.5).astype(int)
calibration["correct"] = calibration["prediction"] == calibration["label"]
calibration["confidence_bin"] = pd.cut(
    calibration["confidence"],
    bins=np.linspace(0.5, 1.0, 11),
    include_lowest=True,
)
reliability = (
    calibration.groupby("confidence_bin", observed=False)
    .agg(
        rows=("label", "size"),
        mean_confidence=("confidence", "mean"),
        empirical_accuracy=("correct", "mean"),
    )
    .reset_index()
)
reliability.to_csv(RESULTS_DIR / "logistic_reliability.csv", index=False)

threshold_audit = []
for threshold in (0.90, 0.95):
    accepted = calibration[calibration["confidence"] > threshold]
    threshold_audit.append({
        "threshold": threshold,
        "accepted": int(len(accepted)),
        "empirical_accuracy": (
            float(accepted["correct"].mean()) if len(accepted) else None
        ),
        "predicted_fake": int((accepted["prediction"] == 0).sum()),
        "predicted_genuine": int((accepted["prediction"] == 1).sum()),
    })
pd.DataFrame(threshold_audit).to_csv(
    RESULTS_DIR / "logistic_threshold_audit.csv",
    index=False,
)


26/09/20 21:58:59 WARN DAGScheduler: Broadcasting large task binary with size 1666.2 KiB
26/09/20 21:59:00 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/09/20 21:59:01 WARN DAGScheduler: Broadcasting large task binary with size 1885.2 KiB
26/09/20 21:59:02 WARN DAGScheduler: Broadcasting large task binary with size 1728.0 KiB
26/09/20 21:59:04 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/09/20 21:59:04 WARN DAGScheduler: Broadcasting large task binary with size 1659.4 KiB


In [5]:
training_audit = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "training_rows": train.count(),
    "validation_rows": validation.count(),
    "class_counts": label_counts,
    "class_weights": class_weights,
    "cross_validated_models": ["logistic_regression", "gbt"],
    "cross_validation_folds": 3,
    "selection": selection,
    "validation_metrics": validation_results,
    "pseudo_label_generator": "logistic_regression",
    "threshold_diagnostics": threshold_audit,
}
with open(AUDIT_DIR / "gold_training_audit.json", "w", encoding="utf-8") as handle:
    json.dump(training_audit, handle, ensure_ascii=False, indent=2)

print(validation_results_frame[
    ["model", "macro_f1", "fake_pr_auc", "fake_f1", "accuracy"]
].to_string(index=False))
print(json.dumps(selection, indent=2))


                model  macro_f1  fake_pr_auc  fake_f1  accuracy
        random_forest  0.925227     0.923503 0.870968  0.964602
  logistic_regression  0.909382     0.889883 0.844560  0.955752
                  gbt  0.907434     0.917321 0.842105  0.953540
        decision_tree  0.864969     0.861550 0.773455  0.926991
           linear_svc  0.775830     0.667670 0.609687  0.898968
          naive_bayes  0.713890     0.539672 0.533066  0.828171
factorization_machine  0.667334     0.345470 0.426667  0.841445
{
  "selected_model": "random_forest",
  "selected_model_type": "random_forest",
  "selection_source": "gold validation split",
  "primary_metric": "macro_f1",
  "tie_break_metric": "fake_pr_auc"
}
